In [ ]:
# Week 8: Customer & Sales Performance Analysis

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Load Datasets
data_path = '../data/'
customers_df = pd.read_csv(os.path.join(data_path, 'customers.csv'))
engineered_sales_df = pd.read_csv(os.path.join(data_path, 'engineered_sales.csv'))

# 2. Aggregate Customer Sales & Order Frequency
cust_metrics = engineered_sales_df.groupby('customer_id').agg(
    total_orders=('order_id', 'nunique'),
    total_units=('quantity', 'sum'),
    total_revenue=('total_revenue', 'sum')
).reset_index()

# 3. Merge Customer Profile Information
cust_analysis = customers_df.merge(cust_metrics, on='customer_id', how='left').fillna(0)

# 4. Customer Value Segmentation (High/Medium/Low-Value)
revenue_75 = cust_analysis['total_revenue'].quantile(0.75)
revenue_25 = cust_analysis['total_revenue'].quantile(0.25)

def segment_customer(rev):
    if rev >= revenue_75:
        return 'High-Value'
    elif rev >= revenue_25:
        return 'Medium-Value'
    else:
        return 'Low-Value'

cust_analysis['customer_value_tier'] = cust_analysis['total_revenue'].apply(segment_customer)

# 5. Save Processed Deliverable
output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)
cust_analysis.to_csv(os.path.join(output_path, 'customer_sales_performance_summary.csv'), index=False)
print('Week 8 customer & sales analysis pipeline completed successfully.')